# 03 · Train seq2seq (encoder-decoder + BPE) để so sánh với tagger

Cần **GPU T4**. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + `data/splits` của notebook 01 (Colab: nằm sẵn trên Drive). Trên Colab mỗi lần train tối đa
150 phút, `runs/s2s/last.pt` lưu trên Drive, chạy lại notebook là train tiếp (`--resume`).
Cùng dữ liệu, số tham số tương đương (3 encoder + 3 decoder layer). Chỉ train 30 phút (tagger 62 phút) cho đỡ tốn giờ GPU.
Mục đích: xem cách nhìn bài toán "dịch máy" có lợi/hại gì so với "gán nhãn ký tự":
chậm hơn bao nhiêu, và có bịa chữ (hallucination) không.

In [ ]:
import glob, json, os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_diacritics/__init__.py", recursive=True)[0].rsplit("/vi_diacritics", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-diacritics-transformer" if IN_COLAB else os.path.abspath("vi-diacritics-transformer")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-diacritics-transformer {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    WORK = os.getcwd()
    if IN_COLAB:
        from google.colab import drive
        try:  # không ai bấm cho phép Drive thì sau 2 phút chạy trên ổ tạm /content (mất khi runtime tắt)
            drive.mount("/content/drive", timeout_ms=120_000)
            WORK = "/content/drive/MyDrive/ai-portfolio/vi-diacritics-transformer"
        except Exception as e:
            print("không mount được Drive, chạy trên /content/work:", e)
            WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


try:
    SPLITS = os.path.dirname(find("splits/train.txt", "chạy notebook 01 trước"))
except FileNotFoundError:
    # Drive chưa có output notebook 01 (vd chạy ở tài khoản khác) -> tự tạo lại, ~10 phút.
    # Chia theo hash của câu nên ra đúng các tập như notebook 01, test vẫn so được với tagger.
    !pip install -q "datasets>=2.19"
    from vi_diacritics.data import build_splits, iter_wikipedia
    print(build_splits(iter_wikipedia(), "data/splits", max_train=2_000_000))
    SPLITS = "data/splits"
# Kaggle cho chạy liền 12h; Colab free hay ngắt sau vài tiếng nên mỗi lần chỉ chạy 150 phút,
# last.pt lưu trên Drive, chạy lại notebook là --resume train tiếp
MAX_MIN = 150 if IN_COLAB else 300
!pip install -q tokenizers

In [ ]:
!python -m vi_diacritics.train --task seq2seq --data {SPLITS} --out runs/s2s \
    --layers 6 --d-model 256 --heads 8 --d-ff 1024 --batch-size 128 --lr 5e-4 \
    --epochs 1 --eval-every 4000 --eval-n 1000 --max-minutes 30 --resume --bpe-vocab 8000

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open("runs/s2s/log.jsonl", encoding="utf-8")]
tr = [r for r in rows if "loss" in r]
ev = [r["eval"] for r in rows if "eval" in r]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["step"] for r in tr], [r["loss"] for r in tr]); ax[0].set_title("train loss")
ax[1].plot([e["step"] for e in ev], [e["word_acc"] for e in ev], label="word acc")
ax[1].plot([e["step"] for e in ev], [e["hallucination_rate"] for e in ev], label="hallucination")
ax[1].legend()
fig.savefig("runs/s2s/curves.png", dpi=110, bbox_inches="tight")

## Vài ví dụ bịa chữ

In [ ]:
from vi_diacritics.dataset import read_lines
from vi_diacritics.evaluate import _seq2seq_fn
from vi_diacritics.labels import strip

f = _seq2seq_fn(Path("runs/s2s"))
test = read_lines(f"{SPLITS}/test.txt", 300)
bad = []
for g in test:
    p = f(strip(g))
    if strip(p) != strip(g):
        bad.append((strip(g), p))
print(f"{len(bad)}/{len(test)} câu bị đổi chữ")
for src, p in bad[:15]:
    print("IN :", src)
    print("OUT:", p)
    print()

## Đánh giá trên tập test, cùng bảng với tagger
Tagger lấy bản ONNX đã đẩy lên HF (notebook 04). Baseline bigram chỉ có khi `baseline.json` nằm trên Drive.
ms/câu: tagger đo ONNX CPU 1 luồng, seq2seq decode greedy trên GPU từng câu một.

In [ ]:
from huggingface_hub import snapshot_download

EXPORT = snapshot_download("hgdkakhs/vi-diacritics-tagger", allow_patterns=["*.onnx", "*.json"], local_dir="export")
if not glob.glob(f"{WORK}/**/nrl_all.txt", recursive=True):
    !pip install -q "datasets>=2.19"
    from datasets import get_dataset_config_names, load_dataset
    from vi_diacritics.data import accent_ratio, normalize_text
    os.makedirs("data/eval", exist_ok=True)
    all_lines = []
    for cfg in get_dataset_config_names("nrl-ai/vn-diacritic-eval"):  # giống notebook 01
        ev = load_dataset("nrl-ai/vn-diacritic-eval", cfg)
        ds = ev[list(ev.keys())[0]]
        text_cols = [c for c in ds.column_names if isinstance(ds[0][c], str)]
        sample = ds.select(range(min(50, len(ds))))
        col = max(text_cols, key=lambda c: sum(accent_ratio(r[c]) for r in sample))
        lines = [normalize_text(r[col]) for r in ds if r[col].strip()]
        open(f"data/eval/nrl_{cfg.rsplit('_', 1)[0]}.txt", "w", encoding="utf-8").writelines(x + "\n" for x in lines)
        all_lines += lines
    open("data/eval/nrl_all.txt", "w", encoding="utf-8").writelines(x + "\n" for x in all_lines)
EVAL = os.path.dirname(glob.glob(f"{WORK}/**/nrl_all.txt", recursive=True)[0])
REAL = f"{REPO}/data/real_typing.txt"
tests = [f"wiki={SPLITS}/test.txt", f"real={REAL}"] + [
    f"{os.path.basename(p)[:-4]}={p}" for p in sorted(glob.glob(f"{EVAL}/nrl_*.txt"))]
args = " ".join(f"--test {t}" for t in tests)
bl = glob.glob(f"{WORK}/baseline.json")
if bl:
    args += f" --baseline {bl[0]}"
!pip install -q onnxruntime
!python -m vi_diacritics.evaluate --export {EXPORT} {args} --s2s-run runs/s2s --limit 3000 --out results_s2s
print(open("results_s2s/eval.md", encoding="utf-8").read())

In [ ]:
# đẩy model seq2seq + kết quả lên HF (thiếu HF_TOKEN thì bỏ qua, số vẫn in ở trên)
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
if token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    HF_REPO = f"{api.whoami()['name']}/vi-diacritics-seq2seq"
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(folder_path="runs/s2s", repo_id=HF_REPO, allow_patterns=["best.pt", "*.json", "*.jsonl", "*.png"])
    api.upload_folder(folder_path="results_s2s", repo_id=HF_REPO, path_in_repo="results")
    print(f"https://huggingface.co/{HF_REPO}")